# G17 — export final + R5 ensembles, calibrate the review flag, publish to Hugging Face
Inputs: **G17_FINAL_model** and **G17_loss_search_R5** outputs. Kaggle secret `HF_TOKEN`/`HF-TOKEN` (write, aee4) → creates `aee4/G17-BUS-Segmentation` (public) and uploads.

In [ ]:
!pip -q install datasets safetensors huggingface_hub segmentation-models-pytorch
import torch; print(torch.__version__, torch.cuda.device_count())


In [ ]:
%%writefile g17core.py
# ============================================================
# Core code: data, model, losses, training, evaluation
# (architecture + split copied from training/training_marimo.py)
# ============================================================
import gc, json, math, os, random, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from scipy.ndimage import binary_erosion, distance_transform_edt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG = int(os.environ.get("G17_IMG", 256))   # model input size; metrics are always computed at EVAL=256
EVAL = 256
MODALITY_COL = {"raw": "raw_image", "clahe": "clahe_image", "srad": "srad_image", "enhanced": "image"}


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


# ---------------- Data: the exact 266-scan test set used by Exp-16–82 ----------------
# StratifiedGroupKFold(shuffle=True) gives different folds on different scikit-learn versions
# (on Kaggle it yields 255 test scans, not 266). So the test set is pinned to the image list
# saved with Exp-16, every patient in it is excluded from train/val, and the rest is split as before.
def load_splits(data_repo="Ahorts/G17-preprocessed-dataset", seed=42,
                model_repo="Ahorts/G17-AMFU-Net-Paper",
                pin_branch="exp16-unet-dag-gpu-reflect-raw-cosine-only", cv_fold=None, n_cv=5):
    from datasets import concatenate_datasets, load_dataset
    from huggingface_hub import hf_hub_download
    from sklearn.model_selection import StratifiedGroupKFold
    hf = load_dataset(data_repo, revision="11b552497ffa661fa16c58c95bd7415b31073721")  # dataset as used for Exp-16–82
    allds = concatenate_datasets([hf["train"], hf["validation"], hf["test"]])
    ref = pd.read_csv(hf_hub_download(model_repo, "results/internal_test_per_image.csv", revision=pin_branch))
    want = set(zip(ref["dataset"].astype(str), ref["image_id"].astype(str)))
    keys = list(zip(np.asarray(allds["dataset"]).astype(str), np.asarray(allds["image_id"]).astype(str)))
    test_idx = np.array([i for i, k in enumerate(keys) if k in want])
    assert len(test_idx) == len(want) == 266, f"pinned test set mismatch: found {len(test_idx)} of {len(want)}"
    groups = np.asarray(allds["patient_id"]).astype(str)
    test_patients = set(groups[test_idx])
    rem = np.array([i for i in range(len(allds)) if groups[i] not in test_patients])
    labels = np.array([f"{d}_{c}" for d, c in zip(allds["dataset"], allds["class_label"])])
    cvt = None
    if cv_fold is not None:  # 5-fold patient-level CV over train+val; the pinned test stays untouched
        folds = list(StratifiedGroupKFold(n_cv, shuffle=True, random_state=seed).split(rem, labels[rem], groups[rem]))
        keep, hold = folds[cv_fold]; cvt = rem[hold]; rem = rem[keep]
    tr, va = next(StratifiedGroupKFold(9, shuffle=True, random_state=seed + 1).split(rem, labels[rem], groups[rem]))
    split = {"train": allds.select(rem[tr].tolist()), "validation": allds.select(rem[va].tolist()),
             "test": allds.select(test_idx.tolist())}
    if cvt is not None:
        split["cvtest"] = allds.select(cvt.tolist())
        assert not set(groups[cvt]) & (set(groups[rem]) | test_patients), "CV patient leakage"
    assert not (set(groups[rem[tr]]) | set(groups[rem[va]])) & test_patients, "patient leakage"
    print({k: len(v) for k, v in split.items()}, "| test pinned to", pin_branch, flush=True)
    return split


_TENSOR_CACHE = {}


def to_tensors(ds, modality, split_name):
    """Decode + resize once per (split, modality), keep uint8 on GPU like the original pipeline."""
    key = (split_name, modality)
    if key in _TENSOR_CACHE:
        return _TENSOR_CACHE[key]
    col = MODALITY_COL[modality]
    imgs, msks, mev = [], [], []
    for r in ds:
        imgs.append(cv2.resize(np.asarray(r[col].convert("L")), (IMG, IMG), interpolation=cv2.INTER_AREA))
        m0 = np.asarray(r["mask"].convert("L"))
        msks.append(cv2.resize(m0, (IMG, IMG), interpolation=cv2.INTER_NEAREST))
        mev.append(cv2.resize(m0, (EVAL, EVAL), interpolation=cv2.INTER_NEAREST))
    out = {"img": torch.from_numpy(np.stack(imgs))[:, None].to(DEVICE),
           "msk": torch.from_numpy(np.stack(msks))[:, None].to(DEVICE),
           "msk_eval": np.stack(mev) > 127,
           "dataset": list(ds["dataset"]), "image_id": list(ds["image_id"])}
    _TENSOR_CACHE[key] = out
    return out


def norm(img_u8):
    return (img_u8.float() / 255.0 - 0.5) / 0.5


def _fan_mask(h, w, device):
    """Random convex-probe sector (fan) mask: narrow at the top, wide at the bottom."""
    top = random.uniform(0.15, 0.6) * w; bot = random.uniform(0.85, 1.3) * w; cx = w / 2 + random.uniform(-0.05, 0.05) * w
    yy, xx = torch.meshgrid(torch.arange(h, device=device), torch.arange(w, device=device), indexing="ij")
    half = (top + (bot - top) * yy / (h - 1)) / 2
    return ((xx - cx).abs() <= half).float()


def appearance_aug(img, msk):
    """Scanner-style appearance changes on a 0..255 float image (C,H,W). Mask only changes for the fan crop."""
    if random.random() < 0.8:                                    # brightness / contrast
        m = img.mean(); img = (img - m) * random.uniform(0.6, 1.4) + m + random.uniform(-30, 30)
    img = img.clamp(0, 255)
    if random.random() < 0.5:                                    # gamma
        img = 255.0 * (img / 255.0).pow(random.uniform(0.6, 1.6))
    if random.random() < 0.5:                                    # multiplicative speckle
        img = img * (1 + torch.randn_like(img) * random.uniform(0.03, 0.18))
    if random.random() < 0.3:                                    # additive noise
        img = img + torch.randn_like(img) * random.uniform(2, 10)
    if random.random() < 0.3:                                    # blur
        k = random.choice([3, 5]); img = TF.gaussian_blur(img, [k, k], [random.uniform(0.3, 1.5)] * 2)
    if random.random() < 0.25:                                   # convex-probe fan framing
        f = _fan_mask(img.shape[-2], img.shape[-1], img.device); img = img * f; msk = msk * f
    return img.clamp(0, 255), msk


def augment(img, msk, strong=False):
    """Same augmentation as training: hflip, rot ±15, scale 0.9–1.1, translate ±10%,
    96-px reflection pad for the image, zero fill for the mask. strong=True adds scanner-appearance changes."""
    if random.random() < 0.5:
        img, msk = TF.hflip(img), TF.hflip(msk)
    angle, scale = random.uniform(-15, 15), random.uniform(0.9, 1.1)
    tx, ty = random.uniform(-0.1, 0.1) * IMG, random.uniform(-0.1, 0.1) * IMG
    p = 96
    imgp = F.pad(img[None].float(), (p, p, p, p), mode="reflect")[0]
    imgp = TF.affine(imgp, angle=angle, translate=[tx, ty], scale=scale, shear=0.0,
                     interpolation=TF.InterpolationMode.BILINEAR)
    img = imgp[:, p:p + IMG, p:p + IMG]
    msk = TF.affine(msk, angle=angle, translate=[tx, ty], scale=scale, shear=0.0,
                    interpolation=TF.InterpolationMode.NEAREST)
    if strong:
        img, msk = appearance_aug(img, msk.float())
    return (img / 255.0 - 0.5) / 0.5, (msk > 127).float()


# ---------------- External set: BrEaST (Poland, 4 scanners) ----------------
BREAST_NPZ = f"/kaggle/working/breast{IMG}.npz"
BREAST_BASE = "/kaggle/working/breast256.npz"   # prepared by the notebook → BrEaST is attached


def prepare_breast(glob_root="/kaggle/input", out=BREAST_NPZ, n_folds=5, seed=42):
    """Load BrEaST once (images + merged *_tumor* masks, skip normals), resize to IMG, assign patient-level folds
    (one scan per patient in BrEaST, so case == patient)."""
    import glob as _g, re as _re
    pngs = [p for p in _g.glob(f"{glob_root}/**/*.png", recursive=True) if "/runs/" not in p]
    imgs = {os.path.splitext(os.path.basename(p))[0]: p for p in pngs if not _re.search(r"_(tumou?r|other|mask)", os.path.basename(p), _re.I)}
    groups = {}
    for m in [p for p in pngs if _re.search(r"_tumou?r", os.path.basename(p), _re.I)]:
        groups.setdefault(_re.split(r"_tumou?r", os.path.splitext(os.path.basename(m))[0], flags=_re.I)[0], []).append(m)
    ids, I, M, ME = [], [], [], []
    for k in sorted(k for k in groups if k in imgs):
        im = cv2.imread(imgs[k], cv2.IMREAD_GRAYSCALE); mk = np.zeros(im.shape, np.uint8)
        for m in groups[k]:
            mm = cv2.imread(m, cv2.IMREAD_GRAYSCALE)
            if mm.shape != im.shape: mm = cv2.resize(mm, (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST)
            mk |= ((mm > 127) * 255).astype(np.uint8)
        if mk.max() == 0: continue
        ids.append(k); I.append(cv2.resize(im, (IMG, IMG), interpolation=cv2.INTER_AREA))
        M.append(cv2.resize(mk, (IMG, IMG), interpolation=cv2.INTER_NEAREST))
        ME.append(cv2.resize(mk, (EVAL, EVAL), interpolation=cv2.INTER_NEAREST))
    rng = np.random.RandomState(seed); order = rng.permutation(len(ids)); fold = np.zeros(len(ids), int)
    for f in range(n_folds): fold[order[f::n_folds]] = f
    tmp = out + f".tmp{os.getpid()}.npz"
    np.savez_compressed(tmp, img=np.stack(I), msk=np.stack(M), msk_eval=np.stack(ME), ids=np.array(ids), fold=fold); os.replace(tmp, out)
    print(f"BrEaST prepared: {len(ids)} lesion scans, folds {np.bincount(fold).tolist()}", flush=True)


def breast_tensors(folds=None, exclude=None):
    z = np.load(BREAST_NPZ, allow_pickle=True); keep = np.ones(len(z["ids"]), bool)
    if folds is not None: keep &= np.isin(z["fold"], folds)
    if exclude is not None: keep &= ~np.isin(z["fold"], exclude)
    return {"img": torch.from_numpy(z["img"][keep])[:, None].to(DEVICE), "msk": torch.from_numpy(z["msk"][keep])[:, None].to(DEVICE),
            "msk_eval": (z["msk_eval"][keep] > 127) if "msk_eval" in z else (z["msk"][keep] > 127),
            "dataset": ["BrEaST"] * int(keep.sum()), "image_id": list(z["ids"][keep])}


def ext_eval(model, name, out, ext, prefix="ext", label="EXT", setname="BrEaST"):
    rows = []; probs = {}
    for v, tta in (("plain", False), ("tta", True)):
        p = predict(model, ext, tta=tta); probs[v] = p
        d = full_metrics(p, ext); d.to_csv(out / f"{prefix}_{v}.csv", index=False); rows.append(summarise(d, name).assign(variant=v))
    np.savez_compressed(out / f"{prefix}_probs.npz", ids=np.array(ext["image_id"]), plain=(probs["plain"] * 255).astype(np.uint8), tta=(probs["tta"] * 255).astype(np.uint8))
    s = pd.concat(rows); s.to_csv(out / f"{prefix}_summary_all.csv", index=False); s = s[s.cohort == "Overall"]; s.to_csv(out / f"{prefix}_summary.csv", index=False)
    print(f"\n### {label} {name} ({setname} n={len(ext['image_id'])})\n" + s[["variant", "dice", "iou", "hd95", "asd", "bf1", "fail_lt50"]].round(2).to_string(index=False), flush=True)


# ---------------- Model ----------------
class ConvBNReLU(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.block = nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


class DAG(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.branch_average = nn.Conv2d(c, c, 3, padding=1)
        self.branch_maximum = nn.Conv2d(c, c, 3, padding=1)
        self.output = nn.Conv2d(c, c, 3, padding=1)

    def forward(self, x):
        g1 = F.adaptive_avg_pool2d(torch.sigmoid(self.branch_average(x)), 1)
        g2 = F.adaptive_max_pool2d(torch.sigmoid(self.branch_maximum(x)), 1)
        return self.output(x * g1 + x * g2)


class MFEF(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 1); self.conv3 = nn.Conv2d(c, c, 3, padding=1)
        self.conv5 = nn.Conv2d(c, c, 5, padding=2); self.gap_projection = nn.Conv2d(c, c, 1)
        self.compress = nn.Conv2d(c * 4, c, 1); self.dilated = nn.Conv2d(c, c, 3, padding=2, dilation=2)

    def forward(self, x):
        g = F.interpolate(self.gap_projection(F.adaptive_avg_pool2d(x, 1)), size=x.shape[-2:],
                          mode="bilinear", align_corners=False)
        w1 = torch.softmax(F.relu(self.compress(torch.cat([self.conv1(x), self.conv3(x), self.conv5(x), g], 1))), 1)
        w2 = torch.softmax(F.relu(self.dilated(x)), 1)
        return x * (0.5 * w1 + 0.5 * w2)


class AMFUNet(nn.Module):
    def __init__(self, base_channels=32, use_dag=True, use_mfef=False, **_):
        super().__init__()
        b = base_channels; self.use_dag, self.use_mfef = use_dag, use_mfef
        self.enc1, self.enc2 = ConvBNReLU(1, b), ConvBNReLU(b, b * 2)
        self.enc3, self.enc4 = ConvBNReLU(b * 2, b * 4), ConvBNReLU(b * 4, b * 8)
        self.bottleneck = ConvBNReLU(b * 8, b * 16); self.pool = nn.MaxPool2d(2)
        if use_mfef:
            self.mfef = MFEF(b * 16)
        if use_dag:
            self.dag1, self.dag2, self.dag3, self.dag4 = DAG(b), DAG(b * 2), DAG(b * 4), DAG(b * 8)
        self.up4, self.dec4 = nn.ConvTranspose2d(b * 16, b * 8, 2, 2), ConvBNReLU(b * 16, b * 8)
        self.up3, self.dec3 = nn.ConvTranspose2d(b * 8, b * 4, 2, 2), ConvBNReLU(b * 8, b * 4)
        self.up2, self.dec2 = nn.ConvTranspose2d(b * 4, b * 2, 2, 2), ConvBNReLU(b * 4, b * 2)
        self.up1, self.dec1 = nn.ConvTranspose2d(b * 2, b, 2, 2), ConvBNReLU(b * 2, b)
        self.head = nn.Conv2d(b, 1, 1)

    def forward(self, x):
        s1 = self.enc1(x); s2 = self.enc2(self.pool(s1)); s3 = self.enc3(self.pool(s2)); s4 = self.enc4(self.pool(s3))
        x = self.bottleneck(self.pool(s4))
        if self.use_mfef:
            x = self.mfef(x)
        for up, dec, s, k in ((self.up4, self.dec4, s4, 4), (self.up3, self.dec3, s3, 3),
                              (self.up2, self.dec2, s2, 2), (self.up1, self.dec1, s1, 1)):
            x = dec(torch.cat([up(x), getattr(self, f"dag{k}")(s) if self.use_dag else s], 1))
        return self.head(x)


def build_model(cfg):
    """AMFU-Net family by default; arch='smp:<Unet|UnetPlusPlus>:<encoder>' uses an ImageNet-pretrained encoder."""
    arch = cfg.get("arch", "amfu")
    if arch.startswith("smp:"):
        import segmentation_models_pytorch as smp
        _, name, enc = arch.split(":")
        return getattr(smp, name)(encoder_name=enc, encoder_weights="imagenet", in_channels=1, classes=1)
    return AMFUNet(use_dag=cfg.get("dag", True), use_mfef=cfg.get("mfef", False))


# ---------------- Losses ----------------
def extract_boundary(t):
    """Differentiable morphological gradient (dilation − erosion), as in training_marimo.py."""
    return F.max_pool2d(t, 3, 1, 1) - (-F.max_pool2d(-t, 3, 1, 1))


def boundary_weight_at(epoch, lam, delay=0, ramp=0):
    """λ schedule: 0 until `delay` epochs, then linear ramp to λ over `ramp` epochs (Ahorts' wu10-20 runs)."""
    if lam <= 0:
        return 0.0
    if epoch < delay:
        return 0.0
    if ramp <= 0:
        return lam
    return lam * min(1.0, (epoch - delay) / ramp)


def compute_loss(logits, labels, cfg, bnd_w):
    logits, labels = logits.float(), labels.float()
    p = torch.sigmoid(logits)
    dims = (1, 2, 3)
    loss = F.binary_cross_entropy_with_logits(logits, labels)
    if cfg["base"] == "dice":                      # BCE + soft Dice (Exp-16 family)
        inter = (p * labels).sum(dims)
        dice = (2 * inter + 1.0) / (p.sum(dims) + labels.sum(dims) + 1.0)
        loss = loss + (1.0 - dice.mean())
    elif cfg["base"] == "ft":                      # BCE + Focal Tversky (Exp-32–47 family)
        a, b, g = cfg["alpha"], cfg["beta"], cfg["gamma"]
        tp = (p * labels).sum(dims); fp = (p * (1 - labels)).sum(dims); fn = ((1 - p) * labels).sum(dims)
        ti = (tp + 1.0) / (tp + a * fp + b * fn + 1.0)
        loss = loss + torch.pow(torch.clamp(1 - ti, 1e-7, 1.0), 1.0 / g).mean()
    if bnd_w > 0 and cfg.get("bnd_type", "morph") == "morph":   # Ahorts-style: soft Dice between boundary maps
        pb, lb = extract_boundary(p), extract_boundary(labels)
        bd = (2 * (pb * lb).sum(dims) + 1.0) / (pb.sum(dims) + lb.sum(dims) + 1.0)
        loss = loss + bnd_w * (1.0 - bd.mean())
    if cfg.get("fpd", 0) > 0:                                   # NEW: distance-weighted false-positive penalty
        d = outside_distance(labels)                             # 0 inside GT, grows with distance outside it
        loss = loss + cfg["fpd"] * ((p * d).sum(dims) / (p.sum(dims) + 1.0)).mean()
    if bnd_w <= 0:
        return loss
    if bnd_w > 0 and cfg.get("bnd_type", "morph") == "morph":
        return loss
    elif bnd_w > 0 and cfg["bnd_type"] == "sdf":                 # Kervadec et al. 2019 distance-map boundary loss
        loss = loss + bnd_w * (p * signed_distance(labels)).mean()
    return loss


def outside_distance(labels):
    """Euclidean distance (fraction of image side) from each pixel to the nearest GT pixel; 0 inside GT."""
    out = np.zeros(labels.shape, np.float32)
    for i, m in enumerate(labels[:, 0].detach().cpu().numpy() > 0.5):
        if m.any():
            out[i, 0] = distance_transform_edt(~m) / m.shape[0]
    return torch.from_numpy(out).to(labels.device)


def signed_distance(labels):
    """Signed distance to the GT contour, normalised by image size: <0 inside, >0 outside."""
    out = np.zeros(labels.shape, np.float32)
    for i, m in enumerate(labels[:, 0].detach().cpu().numpy() > 0.5):
        if m.any() and (~m).any():
            out[i, 0] = (distance_transform_edt(~m) - (distance_transform_edt(m) - 1) * m) / m.shape[0]
    return torch.from_numpy(out).to(labels.device)


# ---------------- Metrics (identical definitions to training) ----------------
def surface_distances(pred, tgt):
    pred, tgt = pred.astype(bool), tgt.astype(bool)
    if not pred.any() and not tgt.any():
        return 0.0, 0.0
    if not pred.any() or not tgt.any():
        d = float(math.hypot(*tgt.shape)); return d, d
    pb = pred ^ binary_erosion(pred, border_value=0); tb = tgt ^ binary_erosion(tgt, border_value=0)
    pb = pb if pb.any() else pred; tb = tb if tb.any() else tgt
    d = np.concatenate([distance_transform_edt(~tb)[pb], distance_transform_edt(~pb)[tb]])
    return float(np.percentile(d, 95)), float(d.mean())


def boundary_f1(pred, tgt, tol=2):
    """Boundary F1 at `tol` px (as in Ahorts' qualitative audit): contour precision/recall within tolerance."""
    pred, tgt = pred.astype(bool), tgt.astype(bool)
    if not pred.any() and not tgt.any():
        return 1.0
    if not pred.any() or not tgt.any():
        return 0.0
    pb = pred ^ binary_erosion(pred, border_value=0); tb = tgt ^ binary_erosion(tgt, border_value=0)
    pb = pb if pb.any() else pred; tb = tb if tb.any() else tgt
    prec = (distance_transform_edt(~tb)[pb] <= tol).mean()
    rec = (distance_transform_edt(~pb)[tb] <= tol).mean()
    return float(2 * prec * rec / (prec + rec + 1e-12))


def confident_cc(mask, prob, keep_prob=0.9, keep_frac=0.2):
    """Smarter filter: keep every component that is either confident (max prob >= keep_prob) or
    at least `keep_frac` of the largest component; drops small low-confidence islands only."""
    from scipy.ndimage import label
    lab, n = label(mask)
    if n <= 1:
        return mask
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    big = sizes.max(); keep = np.zeros(n + 1, bool)
    for k in range(1, n + 1):
        keep[k] = sizes[k] >= keep_frac * big or prob[lab == k].max() >= keep_prob
    return keep[lab]


def largest_cc(mask):
    """Keep only the largest connected component (removes distant false-positive islands)."""
    from scipy.ndimage import label
    lab, n = label(mask)
    if n <= 1:
        return mask
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    return lab == sizes.argmax()


@torch.no_grad()
def predict(model, data, bs=32, tta=False):
    model.eval(); out = []
    for i in range(0, len(data["img"]), bs):
        x = norm(data["img"][i:i + bs])
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            p = torch.sigmoid(model(x).float())
            if tta:
                p = 0.5 * (p + torch.flip(torch.sigmoid(model(torch.flip(x, [-1])).float()), [-1]))
        if p.shape[-1] != EVAL:   # score every model on the same 256-px masks
            p = F.interpolate(p, size=(EVAL, EVAL), mode="bilinear", align_corners=False)
        out.append(p.cpu())
    return torch.cat(out)[:, 0].numpy()


def _gt(data):
    return data["msk_eval"] if "msk_eval" in data else data["msk"][:, 0].cpu().numpy() > 127


def fast_dice(probs, data, thr=0.5):
    pred = probs >= thr; gt = _gt(data)
    tp = (pred & gt).sum((1, 2)); fp = (pred & ~gt).sum((1, 2)); fn = (~pred & gt).sum((1, 2))
    return float(np.mean((2 * tp + 1e-7) / (2 * tp + fp + fn + 1e-7)))


def full_metrics(probs, data, thr=0.5, lcc=False):
    gt_all = _gt(data); rows = []
    for i, (pr, gt) in enumerate(zip(probs >= thr, gt_all)):
        if lcc == "conf":
            pr = confident_cc(pr, probs[i])
        elif lcc:
            pr = largest_cc(pr)
        tp = (pr & gt).sum(); fp = (pr & ~gt).sum(); fn = (~pr & gt).sum()
        hd95, asd = surface_distances(pr, gt)
        rows.append({"image_id": data["image_id"][i], "dataset": data["dataset"][i],
                     "dice": (2 * tp + 1e-7) / (2 * tp + fp + fn + 1e-7),
                     "jaccard": (tp + 1e-7) / (tp + fp + fn + 1e-7),
                     "recall": (tp + 1e-7) / (tp + fn + 1e-7),
                     "precision": (tp + 1e-7) / (tp + fp + 1e-7),
                     "hd95": hd95, "asd": asd, "bf1": boundary_f1(pr, gt)})
    return pd.DataFrame(rows)


VARIANTS = {"plain": (False, False), "lcc": (False, True), "tta": (True, False), "tta+lcc": (True, True),
            "tta+conf": (True, "conf")}


def evaluate_all(model, data):
    """All four post-processing variants; returns {variant: per-image df} and the TTA probability maps."""
    out, probs = {}, {False: predict(model, data), True: predict(model, data, tta=True)}
    for v, (tta, lcc) in VARIANTS.items():
        out[v] = full_metrics(probs[tta], data, lcc=lcc)
    return out, probs


def summarise(df, name):
    rows = []
    for grp, d in [("Overall", df)] + list(df.groupby("dataset")):
        rows.append({"run": name, "cohort": grp, "n": len(d), "dice": d.dice.mean() * 100,
                     "iou": d.jaccard.mean() * 100, "hd95": d.hd95.mean(), "asd": d.asd.mean(),
                     "bf1": d.bf1.mean() * 100 if "bf1" in d else np.nan,
                     "fail_lt50": (d.dice < 0.5).mean() * 100, "dice_sd": d.dice.std() * 100})
    return pd.DataFrame(rows)


# ---------------- Training ----------------
def train_run(cfg, split, out_root, epochs, batch=4, accum=3, lr=1e-3, wd=1e-5, min_lr=1e-5,
              eval_every=250, use_amp=True):
    name = cfg["name"]; out = Path(out_root) / name; out.mkdir(parents=True, exist_ok=True)
    if (out / "test_per_image.csv").exists():
        print(f"[skip] {name} already finished", flush=True); return pd.read_csv(out / "test_per_image.csv")
    seed_everything(cfg.get("seed", 42))
    tr = to_tensors(split["train"], cfg["modality"], "train")
    va = to_tensors(split["validation"], cfg["modality"], "validation")
    te = to_tensors(split["test"], cfg["modality"], "test")
    ext_test = None
    if cfg.get("trainval"):   # final model: validation scans join training; checkpoint = last epoch (val is in-sample)
        tr = {"img": torch.cat([tr["img"], va["img"]]), "msk": torch.cat([tr["msk"], va["msk"]])}
        print(f"[{name}] train+val: {len(tr['img'])} scans (selection = last epoch)", flush=True)
    if os.path.exists(BREAST_BASE) and not os.path.exists(BREAST_NPZ):
        prepare_breast()   # this worker's input size differs from 256
    if os.path.exists(BREAST_NPZ):
        hold = cfg.get("ext_fold")
        ext_test = breast_tensors(folds=[hold]) if hold is not None else breast_tensors()
        if cfg.get("ext_all"):   # all BrEaST joins training → no external evaluation possible
            bt = breast_tensors(); rep = cfg.get("ext_repeat", 2)
            tr = {"img": torch.cat([tr["img"]] + [bt["img"]] * rep), "msk": torch.cat([tr["msk"]] + [bt["msk"]] * rep)}
            print(f"[{name}] + all BrEaST: {len(bt['image_id'])} scans ×{rep} → {len(tr['img'])} training scans", flush=True)
            ext_test = None
        if hold is not None:  # add the other BrEaST folds (patient-disjoint) to training, repeated to give them weight
            bt = breast_tensors(exclude=[hold]); rep = cfg.get("ext_repeat", 2)
            tr = {"img": torch.cat([tr["img"]] + [bt["img"]] * rep), "msk": torch.cat([tr["msk"]] + [bt["msk"]] * rep)}
            print(f"[{name}] + BrEaST train folds: {len(bt['image_id'])} scans ×{rep}; held-out fold {hold}: {len(ext_test['image_id'])}", flush=True)
    model = build_model(cfg).to(DEVICE)
    nparams = sum(p.numel() for p in model.parameters())
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    n = len(tr["img"]); micro_per_epoch = math.ceil(n / batch)
    steps_per_epoch = math.ceil(micro_per_epoch / accum); total = steps_per_epoch * epochs
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total, eta_min=min_lr)
    scaler = torch.amp.GradScaler(enabled=use_amp and DEVICE.type == "cuda")
    best, best_step, step, log = -1.0, 0, 0, []
    print(f"\n=== {name} | {cfg['modality']} | base={cfg['base']} λ={cfg.get('bnd', 0)} | "
          f"params={nparams:,} | {epochs} ep × {steps_per_epoch} steps = {total} steps ===", flush=True)
    t0 = time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(n).tolist()
        bw = boundary_weight_at(ep, cfg.get("bnd", 0.0), cfg.get("bnd_delay", 0), cfg.get("bnd_ramp", 0))
        opt.zero_grad(set_to_none=True)
        for mi in range(micro_per_epoch):
            ids = perm[mi * batch:(mi + 1) * batch]
            pairs = [augment(tr["img"][j], tr["msk"][j], strong=cfg.get("strong_aug", False)) for j in ids]
            x = torch.stack([a for a, _ in pairs]); y = torch.stack([b for _, b in pairs])
            with torch.autocast(DEVICE.type, enabled=use_amp and DEVICE.type == "cuda"):
                logits = model(x)
            loss = compute_loss(logits, y, cfg, bw) / accum
            scaler.scale(loss).backward()
            if (mi + 1) % accum == 0 or mi + 1 == micro_per_epoch:
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
                if step % eval_every == 0 or step == total:
                    vd = fast_dice(predict(model, va), va); model.train()
                    log.append({"step": step, "epoch": ep + 1, "val_dice": vd, "lr": sched.get_last_lr()[0], "bnd_w": bw})
                    if vd > best and not cfg.get("trainval"):
                        best, best_step = vd, step
                        torch.save(model.state_dict(), out / "best.pt")
        sess_h = (time.time() - float(os.environ.get("G17_SESSION_START", t0))) / 3600
        if sess_h > float(os.environ.get("G17_BUDGET_H", 11.0)) - 0.6:   # leave time for evaluation + saving
            print(f"  [{name}] TIME-STOP after epoch {ep+1}: session {sess_h:.2f} h — evaluating best checkpoint", flush=True); break
        if ep == 0 or (ep + 1) % 10 == 0:
            el = time.time() - t0
            print(f"  [{name}] ep {ep+1:>3}/{epochs}  val_dice(best)={best:.4f}@{best_step}  "
                  f"{el/60:.1f} min elapsed, ETA {el/(ep+1)*(epochs-ep-1)/60:.0f} min", flush=True)
    if cfg.get("trainval"):
        best, best_step = float("nan"), step; torch.save(model.state_dict(), out / "best.pt")
    model.load_state_dict(torch.load(out / "best.pt", map_location=DEVICE))
    finish_eval(model, name, out, te, va)
    if "cvtest" in split:
        ext_eval(model, name, out, to_tensors(split["cvtest"], cfg["modality"], "cvtest"), prefix="cv", label="CV", setname=f"held-out fold {cfg.get('cv_fold')}")
    if ext_test is not None:
        ext_eval(model, name, out, ext_test)
    pd.DataFrame(log).to_csv(out / "val_log.csv", index=False)
    json.dump({**cfg, "params": nparams, "best_val_dice": best, "best_step": best_step, "epochs": epochs,
               "minutes": (time.time() - t0) / 60}, open(out / "config.json", "w"), indent=2)
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return pd.read_csv(out / "test_per_image.csv")


def finish_eval(model, name, out, te, va=None):
    """Evaluate test (and val) under every post-processing variant, save CSVs + TTA prob maps, print summary."""
    res, probs = evaluate_all(model, te)
    rows = []
    for v, df in res.items():
        df.to_csv(out / f"test_{v}.csv", index=False)
        rows.append(summarise(df, name).assign(variant=v, split="test"))
    res["plain"].to_csv(out / "test_per_image.csv", index=False)
    np.savez_compressed(out / "test_probs.npz", tta=(probs[True] * 255).astype(np.uint8),
                        plain=(probs[False] * 255).astype(np.uint8))
    if va is not None:
        vres, vprobs = evaluate_all(model, va)
        np.savez_compressed(out / "val_probs.npz", tta=(vprobs[True] * 255).astype(np.uint8),
                            plain=(vprobs[False] * 255).astype(np.uint8))
        for v, df in vres.items():
            rows.append(summarise(df, name).assign(variant=v, split="val"))
    summ = pd.concat(rows); summ.to_csv(out / "summary.csv", index=False)
    o = summ[(summ.cohort == "Overall")][["split", "variant", "dice", "iou", "hd95", "asd", "bf1", "fail_lt50"]]
    print(f"\n### RESULT {name}\n" + o.round(2).to_string(index=False), flush=True)
    t = summ[(summ.split == "test") & (summ.variant == "plain")][["cohort", "dice", "hd95", "asd", "bf1"]]
    print(t.round(2).to_string(index=False), flush=True)
    return summ


# ---------------- Re-evaluate existing Hugging Face checkpoints on the same split ----------------
def eval_hf_checkpoint(name, branch, split, out_root, repo="Ahorts/G17-AMFU-Net-Paper"):
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    out = Path(out_root) / name; out.mkdir(parents=True, exist_ok=True)
    if (out / "summary.csv").exists():
        return pd.read_csv(out / "summary.csv")
    c = json.load(open(hf_hub_download(repo, "config.json", revision=branch)))
    model = AMFUNet(use_dag=c.get("use_dag", True), use_mfef=c.get("use_mfef", False)).to(DEVICE)
    model.load_state_dict(load_file(hf_hub_download(repo, "model.safetensors", revision=branch)), strict=True)
    modality = ("enhanced" if "enhanced" in branch else "clahe" if "clahe" in branch
                else "srad" if "srad" in branch else "raw")
    te = to_tensors(split["test"], modality, "test")
    return finish_eval(model, name, out, te)   # no val: old checkpoints were selected on a different val split


# ---------------- Ensembles: average TTA probability maps of several runs ----------------
def ensemble_eval(name, members, out_root, split, modality="raw"):
    out = Path(out_root) / name; out.mkdir(parents=True, exist_ok=True)
    rows = []
    for sp, key in (("test", "test"), ("val", "validation")):
        files = [Path(out_root) / m / f"{sp}_probs.npz" for m in members]
        if not all(f.exists() for f in files):
            continue
        arrs = [np.load(f)["tta"] for f in files]
        data = to_tensors(split[key], modality, key)
        if len({a.shape for a in arrs}) > 1 or arrs[0].shape[0] != len(data["image_id"]):
            print(f"[{name}] skip {sp}: members were scored on different {sp} sets (e.g. CV folds)", flush=True); continue
        p = np.mean([a.astype(np.float32) / 255.0 for a in arrs], axis=0)
        for v, lcc in (("tta", False), ("tta+conf", "conf"), ("tta+lcc", True)):
            df = full_metrics(p, data, lcc=lcc)
            if sp == "test":
                df.to_csv(out / f"test_{v}.csv", index=False)
            rows.append(summarise(df, name).assign(variant=v, split=sp))
        if sp == "test":
            np.savez_compressed(out / "test_probs.npz", tta=(p * 255).astype(np.uint8), plain=(p * 255).astype(np.uint8))
            full_metrics(p, data).to_csv(out / "test_plain.csv", index=False)
    summ = pd.concat(rows); summ.to_csv(out / "summary.csv", index=False)
    o = summ[summ.cohort == "Overall"][["split", "variant", "dice", "iou", "hd95", "asd", "bf1", "fail_lt50"]]
    print(f"\n### RESULT {name} = mean({', '.join(members)})\n" + o.round(2).to_string(index=False), flush=True)
    return summ


# ---------------- Parallel workers: one process per GPU ----------------
def worker(gpu, run_names, plan_path, out_root, epochs):
    plan = {c["name"]: c for c in json.load(open(plan_path))}
    split, cur = None, "unset"
    t_start = float(os.environ.get("G17_SESSION_START", time.time()))
    budget_h = float(os.environ.get("G17_BUDGET_H", 11.0))
    for n in run_names:
        est_h = plan[n].get("est_h", 2.5)
        if (time.time() - t_start) / 3600 + est_h > budget_h:
            print(f"[SKIP-TIME] {n}: not enough session time left (est {est_h} h)", flush=True); continue
        try:
            if plan[n].get("cv_fold") != cur:   # each CV fold has its own train/val → reload split, drop cached tensors
                cur = plan[n].get("cv_fold"); _TENSOR_CACHE.clear(); gc.collect(); torch.cuda.empty_cache()
                split = load_splits(cv_fold=cur)
            train_run(plan[n], split, out_root, epochs=plan[n].get("epochs", epochs))
        except Exception as e:  # keep the other runs going
            import traceback; traceback.print_exc(); print(f"[FAILED] {n}: {e}", flush=True)


# ---------------- Ahorts-style contour overlays ----------------
def overlay_figure(out_root, run_names, split, case_ids, path, modality_of):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    te_raw = to_tensors(split["test"], "raw", "test")
    idx = [te_raw["image_id"].index(c) for c in case_ids]
    fig, axes = plt.subplots(len(idx), len(run_names), figsize=(2.6 * len(run_names), 2.6 * len(idx)), squeeze=False)
    gt_all = te_raw["msk"][:, 0].cpu().numpy() > 127
    per = {r: pd.read_csv(Path(out_root) / r / "test_plain.csv").set_index("image_id") for r in run_names}
    for c, r in enumerate(run_names):
        probs = np.load(Path(out_root) / r / "test_probs.npz")["plain"]
        for k, i in enumerate(idx):
            ax = axes[k, c]; ax.imshow(te_raw["img"][i, 0].cpu().numpy(), cmap="gray"); ax.axis("off")
            ax.contour(gt_all[i], levels=[0.5], colors="lime", linewidths=1.2)
            ax.contour(probs[i] >= 128, levels=[0.5], colors="magenta", linewidths=1.0)
            m = per[r].loc[te_raw["image_id"][i]]
            ax.set_title(f"{r[:18]}\nD {m.dice*100:.0f}  HD {m.hd95:.0f}  BF1 {m.bf1*100:.0f}", fontsize=7)
    fig.suptitle("green = ground truth, magenta = prediction", fontsize=9)
    plt.tight_layout(); plt.savefig(path, dpi=75); plt.close(fig)


if __name__ == "__main__":
    import sys
    worker(int(sys.argv[1]), sys.argv[2].split(","), sys.argv[3], sys.argv[4], int(sys.argv[5]))


In [ ]:
%%writefile uncertainty.py
"""Per-scan quality signals from the ensemble, and the calibrated "check this scan" rule.

All signals are computed on the 256 x 256 model grid from each member's flip-averaged probability map.
The rule (metric, direction, threshold) is fitted on the held-out internal test set and shipped with the
weights, so the app never invents its own cut-off.
"""
import numpy as np
from scipy.ndimage import label

SIGNALS = ("agreement", "ambiguity", "mean_confidence", "components", "area_fraction")


def _dice(a, b):
    total = a.sum() + b.sum()
    return 1.0 if total == 0 else float(2 * (a & b).sum() / total)


def quality_signals(member_probs, threshold=0.5):
    """member_probs: (members, H, W) probabilities. Returns a dict of scalar signals."""
    member_probs = np.asarray(member_probs, dtype=np.float32)
    mean = member_probs.mean(0)
    mask = mean >= threshold
    members = member_probs >= threshold
    pairs = [_dice(members[i], members[j]) for i in range(len(members)) for j in range(i + 1, len(members))]
    area = int(mask.sum())
    return {
        "agreement": float(np.mean(pairs)) if pairs else 1.0,        # mean pairwise Dice between member masks
        "ambiguity": float(((mean > 0.3) & (mean < 0.7)).sum() / max(area, 1)),  # uncertain band / mask area
        "mean_confidence": float(mean[mask].mean()) if area else 0.0,
        "components": int(label(mask)[1]),
        "area_fraction": float(area / mask.size),
    }


def needs_review(signals, rule):
    """True when the scan should be checked by a person. An empty mask is always flagged."""
    if signals["area_fraction"] == 0:
        return True
    value = signals[rule["metric"]]
    return value < rule["threshold"] if rule["direction"] == "below" else value > rule["threshold"]


In [ ]:
import glob, json, os, hashlib, numpy as np, pandas as pd, torch, torch.nn.functional as F
import g17core as g, uncertainty as U
from safetensors.torch import save_file, load_file
ckpts = sorted(glob.glob("/kaggle/input/**/runs/FINAL_s*/best.pt", recursive=True))
print(ckpts); assert len(ckpts) == 3, "attach the G17_FINAL_model notebook output as Input"
ROOT = "/kaggle/working/hf_upload"; OUT = f"{ROOT}/final"; os.makedirs(OUT, exist_ok=True)
models, files = [], []
for c in ckpts:
    seed = c.split("FINAL_s")[1].split("/")[0]
    m = g.build_model(dict(arch="smp:UnetPlusPlus:resnet34")).to(g.DEVICE)
    m.load_state_dict(torch.load(c, map_location=g.DEVICE)); m.eval(); models.append(m)
    sd = {k: v.detach().cpu().contiguous() for k, v in m.state_dict().items()}
    fn = f"member_s{seed}.safetensors"; save_file(sd, f"{OUT}/{fn}")
    # round-trip check
    m2 = g.build_model(dict(arch="smp:UnetPlusPlus:resnet34")); m2.load_state_dict(load_file(f"{OUT}/{fn}"), strict=True)
    files.append(fn)
# R5 paper models (best held-out results): attach the G17_loss_search_R5 output as a second Input
R5 = f"{ROOT}/r5_paper"; os.makedirs(R5, exist_ok=True); r5_files = []
for c in sorted(glob.glob("/kaggle/input/**/runs/R5_aug_s*/best.pt", recursive=True)):
    m = g.build_model(dict(arch="smp:UnetPlusPlus:resnet34")); m.load_state_dict(torch.load(c, map_location="cpu"))
    fn = "member_" + c.split("runs/")[1].split("/")[0].replace("R5_aug_", "") + ".safetensors"
    save_file({k: v.contiguous() for k, v in m.state_dict().items()}, f"{R5}/{fn}"); r5_files.append(fn)
print("R5 paper members:", r5_files)
split = g.load_splits(); te = g.to_tensors(split["test"], "raw", "test")


In [ ]:
@torch.no_grad()
def member_probs(model, data, bs=16):
    out = []
    for i in range(0, len(data["img"]), bs):
        x = g.norm(data["img"][i:i + bs])
        p = torch.sigmoid(model(x).float()); p = 0.5 * (p + torch.flip(torch.sigmoid(model(torch.flip(x, [-1])).float()), [-1]))
        out.append(p[:, 0].cpu())
    return torch.cat(out).numpy()
P = np.stack([member_probs(m, te) for m in models], 1)          # (N, 3, 256, 256)
ens = P.mean(1)
d = g.full_metrics(ens, te)                                      # ensemble per-image metrics (same as ENS_FINAL_x3 tta)
print("ensemble test dice", 100 * d.dice.mean())
S = pd.DataFrame([U.quality_signals(P[i]) for i in range(len(P))]); S["dice"] = d.dice.values; S["dataset"] = d.dataset.values
S.to_csv(f"{OUT}/test_quality_signals.csv", index=False)


In [ ]:
from sklearn.metrics import roc_auc_score
rows = []
for metric, sign in (("agreement", -1), ("ambiguity", 1), ("mean_confidence", -1), ("components", 1)):
    for cut in (0.5, 0.7):
        y = (S.dice < cut).astype(int)
        if y.sum() >= 2: rows.append(dict(metric=metric, cut=cut, positives=int(y.sum()), auroc=roc_auc_score(y, sign * S[metric])))
A = pd.DataFrame(rows); print(A.round(3).to_string(index=False))
best = A[A.cut == 0.7].sort_values("auroc", ascending=False).iloc[0].metric
sign = dict(agreement=-1, ambiguity=1, mean_confidence=-1, components=1)[best]
# threshold: flag ~12% of scans (review budget), report what it catches
vals = S[best].values
thr = float(np.quantile(vals, 0.12) if sign < 0 else np.quantile(vals, 0.88))
rule = dict(metric=best, direction="below" if sign < 0 else "above", threshold=round(thr, 4))
flag = np.array([U.needs_review(dict(r), rule) for r in S.to_dict("records")])
print("RULE", rule)
print(f"flagged {flag.mean()*100:.1f}% of test scans | catches {((S.dice<0.5)&flag).sum()}/{(S.dice<0.5).sum()} with dice<0.5, "
      f"{((S.dice<0.7)&flag).sum()}/{(S.dice<0.7).sum()} with dice<0.7 | dice flagged {100*S.dice[flag].mean():.1f} vs unflagged {100*S.dice[~flag].mean():.1f}")


In [ ]:
cfg = dict(architectures=["UnetPlusPlus"], library="segmentation_models_pytorch", encoder_name="resnet34", in_channels=1, classes=1,
           image_size=256, input="grayscale, resize 256x256 INTER_AREA, (x/255-0.5)/0.5, no SRAD/CLAHE",
           members=files, tta="horizontal flip", ensemble="mean of member probabilities", threshold=0.5,
           review_rule=rule, review_rule_calibration=dict(test_scans=int(len(S)), flagged_pct=round(float(flag.mean()*100), 1),
               auroc=A.round(3).to_dict("records")),
           internal_test=dict(scans=int(len(S)), dice=round(float(100*d.dice.mean()), 2), hd95=round(float(d.hd95.mean()), 2),
               bf1=round(float(100*d.bf1.mean()), 2)),
           training="R5 recipe: Focal Tversky a0.5/b0.5 g1.33, strong appearance aug, 100 ep; train+val (2371) + BrEaST x2 (504); last epoch")
cfg["sha256"] = {f: hashlib.sha256(open(f"{OUT}/{f}", "rb").read()).hexdigest() for f in files}
json.dump(cfg, open(f"{OUT}/config.json", "w"), indent=2); print(json.dumps(cfg, indent=1)[:3000])
if r5_files:
    json.dump(dict(architectures=["UnetPlusPlus"], library="segmentation_models_pytorch", encoder_name="resnet34", in_channels=1,
                   classes=1, image_size=256, members=r5_files, tta="horizontal flip", ensemble="mean of member probabilities",
                   sha256={f: hashlib.sha256(open(f"{R5}/{f}", "rb").read()).hexdigest() for f in r5_files},
                   internal_test=dict(scans=266, dice_tta_ensemble=89.36, hd95=11.03), breast_zero_shot=dict(scans=252, dice_tta_ensemble=79.66),
                   cv_5fold=dict(dice="86.38 +/- 0.82"),
                   training="R5 recipe on train (2102) only; checkpoint = best validation Dice; BrEaST never seen (reported model)"),
              open(f"{R5}/config.json", "w"), indent=2)
open(f"{ROOT}/README.md", "w").write(f"""---
license: mit
library_name: segmentation-models-pytorch
tags: [medical-imaging, breast-ultrasound, segmentation, unet-plus-plus]
---
# G17 breast-ultrasound lesion segmentation — final models

Two 3-member U-Net++ (ImageNet ResNet-34 encoder, 1-channel input) ensembles, trained with Focal Tversky loss (a=b=0.5, g=1.33)
and scanner-appearance augmentation, 100 epochs. Input: grayscale, resized to 256x256 (INTER_AREA), (x/255-0.5)/0.5.
Inference: each member averages the image and its horizontal mirror; the ensemble averages member probabilities; threshold 0.5.

| folder | trained on | internal test Dice (266 held-out scans) | BrEaST | use |
|---|---|---|---|---|
| `final/` | train + val + all BrEaST (2875 scans), last epoch | {100*d.dice.mean():.2f} | in training | deployment (G17 Streamlit app) |
| `r5_paper/` | train (2102), best-validation checkpoint | 89.36 | 79.66 zero-shot | reported results |

`final/config.json` also carries the calibrated `review_rule` used by the app's Check flag.
Research use only — not a medical device.
""")
try:
    from kaggle_secrets import UserSecretsClient
    token = None
    for _name in ("HF_TOKEN", "HF-TOKEN"):
        try:
            token = UserSecretsClient().get_secret(_name); break
        except Exception:
            pass
    if not token: raise RuntimeError("secret HF_TOKEN / HF-TOKEN not found")
except Exception as e:
    token = None; print("no HF_TOKEN secret — files are in the notebook output only:", e)
if token:
    from huggingface_hub import HfApi
    api = HfApi(token=token); REPO = "aee4/G17-BUS-Segmentation"; print("token user:", api.whoami()["name"])
    api.create_repo(REPO, repo_type="model", private=False, exist_ok=True)
    info = api.upload_folder(repo_id=REPO, folder_path=ROOT, commit_message="G17 final ensemble (app) + R5 paper ensemble")
    print("PUSHED_REPO", REPO, "COMMIT", info.oid)
